In [9]:
pip install -r requirements.txt


In [2]:
# 2️⃣ Imports and Setup
# -------------------------------
import re
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from transformers import pipeline
import tweepy
import os
from wordcloud import WordCloud
from dotenv import load_dotenv

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Load .env if you have one
load_dotenv()
TWITTER_BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN")

In [4]:
# 3️⃣ Tweet Cleaning Function
# -------------------------------
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove mentions
    text = re.sub(r'#', '', text)                         # remove hashtag symbol
    text = re.sub(r'[^\w\s]', '', text)                  # remove punctuation
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [5]:
# -------------------------------
# 4️⃣ Connect to Twitter API
# -------------------------------
# Replace with your own keys if not using .env
api_key = os.getenv("TWITTER_API_KEY") or "YOUR_API_KEY"
api_secret = os.getenv("TWITTER_API_SECRET") or "YOUR_API_SECRET"
access_token = os.getenv("TWITTER_ACCESS_TOKEN") or "YOUR_ACCESS_TOKEN"
access_token_secret = os.getenv("TWITTER_ACCESS_TOKEN_SECRET") or "YOUR_ACCESS_TOKEN_SECRET"

auth = tweepy.OAuthHandler(api_key, api_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth)


In [ ]:
# -------------------------------
# 5️⃣ Fetch Live Tweets
# -------------------------------
import tweepy
import pandas as pd
import os

BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN") or "YOUR_BEARER_TOKEN"
client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

# Test search
tweets = client.search_recent_tweets(query="Python -is:retweet lang:en", max_results=5)
for t in tweets.data:
    print(t.text)



In [ ]:
# -------------------------------
# 6️⃣ Load Sentiment Model
# -------------------------------
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")


In [ ]:
# -------------------------------
# 7️⃣ Predict Sentiment
# -------------------------------
df['sentiment'] = df['clean_text'].apply(lambda x: sentiment_model(x[:512])[0]['label'])
df['score'] = df['clean_text'].apply(lambda x: sentiment_model(x[:512])[0]['score'])
df.head()


In [ ]:
# -------------------------------
# 8️⃣ Visualize Sentiment Distribution
# -------------------------------
plt.figure(figsize=(6,4))
df['sentiment'].value_counts().plot(kind='bar', color=['tomato','skyblue'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()


In [ ]:
# -------------------------------
# 9️⃣ Word Clouds by Sentiment
# -------------------------------
for label in df['sentiment'].unique():
    text = " ".join(df[df['sentiment']==label]['clean_text'].astype(str).tolist())
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud for {label} Tweets")
    plt.show()


In [ ]:
# -------------------------------
# 🔟 Test Your Own Tweet
# -------------------------------
text = input("Enter a tweet: ")
cleaned = clean_tweet(text)
result = sentiment_model(cleaned[:512])[0]
print(f"\nOriginal: {text}")
print(f"Cleaned: {cleaned}")
print(f"Predicted Sentiment: {result['label']} (Score: {result['score']:.2f})")


In [ ]:
# -------------------------------
# 1️⃣1️⃣ Save Results (Optional)
# -------------------------------
df.to_csv("tweet_sentiment_results.csv", index=False)
print("Results saved to tweet_sentiment_results.csv")
